# ✉️ Messages
  <img src="../assets/LC_Messages.png" width="500">

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd()
while project_root != project_root.parent and not (
    project_root / "utils" / "env_utils.py"
).is_file():
    project_root = project_root.parent

if not (project_root / "utils" / "env_utils.py").is_file():
    raise FileNotFoundError("Could not locate utils/env_utils.py from the notebook directory.")

os.chdir(project_root)
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from utils.env_utils import doublecheck_env, doublecheck_pkgs

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")  # check environmental variables
doublecheck_pkgs(pyproject_path="pyproject.toml", verbose=True)   # check packages

Did not find file example.env.
This is used to double check the key settings for the notebook.
This is just a check and is not required.

Python 3.13.5 satisfies requires-python: >=3.11, <3.14
package                | required | installed | status | path                                                                            
---------------------- | -------- | --------- | ------ | --------------------------------------------------------------------------------
jupyter                | >=1.0.0  | 1.1.1     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain              | >=1.3.10 | 1.4.3     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-anthropic    | >=1.4.6  | 1.7.5     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/python3.13/site-packages
langchain-community    | >=0.4.2  | 0.4.2     | ✅ OK   | …ai-certification-gen-academy/langchain-basic/.venv/lib/pyth

OPENAI_API_KEY=****eJgA

LANGSMITH_API_KEY=****2eed

LANGSMITH_TRACING=true

LANGSMITH_PROJECT=****ject

## Human👨‍💻 and AI 🤖 Messages

In [2]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="openai:gpt-5-nano", 
    system_prompt="You are a full-stack comedian"
)

In [5]:
human_msg = HumanMessage("Hello, how are you?")

result = agent.invoke({"messages": [human_msg]})

In [6]:
print(result["messages"][-1].content)

Hi there! I’m doing well, thanks—powered by coffee and clever puns. I’m a full-stack comedian: front-end humor for instant smiles, back-end banter for the deeper punchlines, and a database of jokes ready to query. Want a quick joke or need help with something?


In [7]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [8]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hi there! I’m doing well, thanks—powered by coffee and clever puns. I’m a full-stack comedian: front-end humor for instant smiles, back-end banter for the deeper punchlines, and a database of jokes ready to query. Want a quick joke or need help with something?



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [9]:
agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [10]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Baseball: dirt, sun, and a heartbeat.
Nine innings—time folded in a glove.
Windup, whisper, bat’s reply.
Crack—wood wakes the summer air.
The ball climbs, or dies in the glove.
Three strikes, three outs, the chalk redraws the diamond.
Walk-offs, double plays, legends born between pitches.
Baseball: memory in motion, named by the pitch.


#### Dictionaries

In [11]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Sprinters whisper fast
Breaths tally, feet drum in time
Finish line whispers


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [12]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [13]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [14]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Dirt clings to cleats' bite
Wind lifts the long fly to dusk
Home roars, glove snaps shut


In [15]:
result["messages"][-1].content

"Dirt clings to cleats' bite\nWind lifts the long fly to dusk\nHome roars, glove snaps shut"

In [16]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (call_6CmGVCDySY8VeMRKtr76hbtv)
 Call ID: call_6CmGVCDySY8VeMRKtr76hbtv
  Args:
    text: Dirt clings to cleats' bite
Wind lifts the long fly to dusk
Home roars, glove snaps shut
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Dirt clings to cleats' bite
Wind lifts the long fly to dusk
Home roars, glove snaps shut


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [17]:
result

{'messages': [HumanMessage(content='Please write me a poem', additional_kwargs={}, response_metadata={}, id='e9facd10-6ae2-4c0c-a8ef-8bb36848cf10'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 882, 'prompt_tokens': 170, 'total_tokens': 1052, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EWmEcrjAxOiuVjNTr2fgxlm9GvXxu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a11c90-d912-7052-b1d3-9e9b5e0f738f-0', tool_calls=[{'name': 'check_haiku_lines', 'args': {'text': "Dirt clings to cleats' bite\nWind lifts the long fly to d

You can select just the last message, and you can see where the final message is coming from.

In [18]:
result["messages"][-1]

AIMessage(content="Dirt clings to cleats' bite\nWind lifts the long fly to dusk\nHome roars, glove snaps shut", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 238, 'total_tokens': 266, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EWmEp2gmNCf7Nt7JgxISE8QhylTJS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a11c91-0d5e-7c60-bd9d-ab061a43e18c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 238, 'output_tokens': 28, 'total_tokens': 266, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det

In [19]:
result["messages"][-1].usage_metadata

{'input_tokens': 238,
 'output_tokens': 28,
 'total_tokens': 266,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [20]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 28,
  'prompt_tokens': 238,
  'total_tokens': 266,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0,
   'text_tokens': None},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': None,
   'cached_tokens': 0,
   'image_tokens': None,
   'text_tokens': None}},
 'model_provider': 'openai',
 'model_name': 'gpt-5-2025-08-07',
 'system_fingerprint': None,
 'id': 'chatcmpl-EWmEp2gmNCf7Nt7JgxISE8QhylTJS',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}